## Data representation and prevention of data leakage

The original datasets describe each match from the perspective of the winner and loser. They therefore contain columns such as `Winner`, `Loser`, `WRank`, `LRank`, `WPts`, `LPts`, and bookmaker odds associated with the winner and loser.

Using these columns directly as model inputs would be inappropriate because the labels `Winner` and `Loser` encode the outcome of the match. To avoid this issue, each match is reformulated into a neutral representation:

- `Player1`, `Player2`
- `Player1Rank`, `Player2Rank`
- `Player1Points`, `Player2Points`
- corresponding pre-match odds
- `Player1Won` as the target variable

For approximately half of the matches, Player 1 is assigned to the original winner; for the remaining matches, Player 1 is assigned to the original loser. Consequently, the position of a player in the dataset does not reveal the match outcome.

**Features excluded from prediction**

Only information that could have been known before the match should be used by the prediction models. Features generated during or after the match are therefore excluded.

| Column group | Treatment |
|---|---|
| `Winner`, `Loser` | Used to construct the target and player-specific variables, but not directly as model inputs |
| `W1–W5`, `L1–L5` | Removed because set scores are known only after the match |
| `Wsets`, `Lsets` | Removed because they describe the final result |
| `Comment` | Excluded from prediction because it may reveal retirement, walkover, or other post-match information |
| `SourceFile` / `SourceYear` | Retained for data provenance and analysis, but not intended as a predictive feature |
| `WRank`, `LRank` | Retained after mapping them to Player 1 and Player 2 |
| `WPts`, `LPts` | Retained after mapping them to Player 1 and Player 2 |
| Betting odds | Pre-match information; retained later |
| `Surface`, `Court`, `Series`, `Round`, `Best of` | Valid pre-match information |

Using match scores or post-match comments as predictors would constitute data leakage, because the model would receive information produced by the event it is supposed to predict.

## 02 — Data Preparation

This notebook combines the yearly datasets, standardizes their structure and data types, removes unsuitable observations, and constructs the final neutral match representation used by the subsequent modelling stages.

In [1]:
from pathlib import Path

import numpy as np 
import pandas as pd

In [2]:
RAW_DATA_DIR = Path("../data/raw") # where the original Excel files are
PROCESSED_DATA_DIR = Path("../data/processed") # where the cleaned dataset will be saved

PROCESSED_DATA_DIR.mkdir(parents = True, exist_ok=True) 
# create the processed-data directory if it does not already exist.

print("Raw data folder:", RAW_DATA_DIR.resolve())
print("Processed data folder:", PROCESSED_DATA_DIR.resolve())

Raw data folder: C:\Users\SegreteriaCampiello\Documents\GitHub\artificial_intelligence\data\raw
Processed data folder: C:\Users\SegreteriaCampiello\Documents\GitHub\artificial_intelligence\data\processed


### Column selection across yearly datasets

The older datasets contain bookmaker columns such as `EXW`, `EXL`, `LBW`, and `LBL`, whereas the 2025 dataset contains `BFEW` and `BFEL`. Because these columns are not consistently available across all years, they are not included in the common table.

The bookmaker columns retained here are those available throughout the selected period:

- `B365W`, `B365L`
- `PSW`, `PSL`
- `MaxW`, `MaxL`
- `AvgW`, `AvgL`

At this stage, these variables are preserved for possible later experiments; their inclusion in the main predictive model is decided separately.

The variables `W1–W5`, `L1–L5`, `Wsets`, and `Lsets` are deliberately excluded because they are generated during or after the match and therefore cannot be used for a genuine pre-match prediction.

In [3]:
# we store the years of the datasets we intend to use
years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

## We selected the columns that were in common to all years. 
## We choose them based on the result obtained in the 01_data_understanding notebook
selected_columns = [
    "ATP", 
    "Location", 
    "Tournament", 
    "Date", 
    "Series", 
    "Court", 
    "Surface",
    "Round", 
    "Best of", 
    "Winner", 
    "Loser", 
    "WRank", 
    "LRank", 
    "WPts", 
    "LPts", 
    "Comment", 
    "B365W", 
    "B365L", 
    "PSW",
    "PSL",
    "MaxW",
    "MaxL",
    "AvgW",
    "AvgL" ]

yearly_frames is initalized as an empty list. During the for loop, one cleaned DataFrame will be created and added to this list for each year.

For each dataset:
1. the corresponding Excel file is loaded

2. current_year.columns.str.strip() removes spaces at the beginning or end of a column name, this protects the model against subtle problems caused by accidental spaces.

3. The missing_column list checks whether every column we need is present.

4. With .copy(), we create a separate DataFrame rather than a potentially ambiguous view of the original one.

5. SourceYear records the year from which each row originated.
For example, rows loaded from 2017.xlsx receive:
SourceYear = 2017
This is just useful for checking the data.

--> pd.concat(...) places the yearly tables underneath one another.
--> ignore_index=True creates a new continuous row index: 0, 1, 2, 3, ...

The number of rows should initially be close to the total we already observed: 27,574.

In [4]:
yearly_frames = []


for year in years: 
    # for each dataset the corresponding excel file is loaded
    file_path = RAW_DATA_DIR / f"{year}.xlsx"
    
    current_year = pd.read_excel(file_path)
    
    # columns are stripped of accidental leading or trailing spaces
    current_year.columns = current_year.columns.str.strip()

    # we check if every column we need is present
    # only the selected columns are retained
    missing_columns = [
        column
        for column in selected_columns 
        if column not in current_year.columns 
    ]

    # stops with an error in case there are missing columns
    if missing_columns: 
        raise ValueError(
            f"The {year} dataset is missing these columns: {missing_columns}"
        )

    # creates a separate dataframe
    current_year = current_year[selected_columns].copy()

    # record the file from which every row originated 
    current_year["SourceYear"] = year

    yearly_frames.append(current_year)  

# yearly dataframes are combined, placing their rows one below another
matches = pd.concat(yearly_frames, ignore_index=True)

print("Combined dataset shape:", matches.shape)


c:\Users\SegreteriaCampiello\Documents\GitHub\artificial_intelligence\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\SegreteriaCampiello\Documents\GitHub\artificial_intelligence\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Combined dataset shape: (27574, 25)


In [5]:
print("Number of rows:", len(matches))
print("Number of columns:", len(matches.columns))

matches.head()

matches["SourceYear"].value_counts().sort_index()

Number of rows: 27574
Number of columns: 25


SourceYear
2015    2630
2016    2626
2017    2633
2018    2637
2019    2610
2020    1267
2021    2489
2022    2632
2023    2703
2024    2703
2025    2644
Name: count, dtype: int64

**Data-type standardization**

The yearly Excel files may represent the same kind of information differently.
For example, one ranking might be stored as the number: 54
while another file might accidentally store it as text: "54"

Standardizing these types is necessary so that the combined dataset can be processed consistently by the subsequent data_preparation and modelling steps.

pd.to_datetime(...) covnerts valid values to datetime objects and replaces invalid values with NaT (errors="coerce")

For text variables, values are converted to pandas string type and surrounding whitespace is removed.

For numerical variables, pd.to_numeric(...) coverts valid values to numeric form. Values that cannot be converted are replaced with NaN (errors="coerce").
This is preferable to silently leaving mixed text and numerical values in the same column.

In [6]:
matches["Date"] = pd.to_datetime( # converted to datetime objects
    matches["Date"], 
    errors="coerce"
)

text_columns = [
    "Location",
    "Tournament",
    "Series",
    "Court",
    "Surface",
    "Round",
    "Winner",
    "Loser",
    "Comment"
]

for column in text_columns: 
    matches[column] = matches[column].astype("string").str.strip()
    
numeric_columns = [
    "ATP",
    "Best of",
    "WRank",
    "LRank",
    "WPts",
    "LPts",
    "B365W",
    "B365L",
    "PSW",
    "PSL",
    "MaxW",
    "MaxL",
    "AvgW",
    "AvgL"  
]

for column in numeric_columns: 
    matches[column] = pd.to_numeric( # converted to numerical form
        matches[column], 
        errors="coerce"
    )

After standardizing we check the essential columns for missing values. 

The resulting counts provide a direct check of how many observations contain incomplete information in these fields.

In [7]:
essential_columns = [
    "Date", 
    "Winner", 
    "Loser", 
    "WRank", 
    "LRank", 
    "WPts",
    "LPts"
]

print(matches[essential_columns].isna().sum())

Date       0
Winner     0
Loser      0
WRank     11
LRank     58
WPts      10
LPts      58
dtype: int64


In [8]:
# counts number of matches for each status
print(matches["Comment"].value_counts(dropna=False))

Comment
Completed       26602
Retired           794
Walkover          170
Awarded             5
Sched               1
Disqualified        1
Rrtired             1
Name: count, dtype: Int64


**Comment describes the status of the match**

A walkover is not a normally played match, while a retirement may be caused by an event such as an injury occurring during the match. The available pre-match features describe player strength and previous performance, but they do not contain information that can reliably predict such unexpected events.

For the first version of the project, only completed matches are therefore retained. This creates a clearer and more homogeneous prediction problem.

In [9]:
# we store the number of matches before filtering for comparison reasons
rows_before_status_filter = len(matches)

matches = matches.loc[
    matches["Comment"].eq("Completed")
].copy() # we only keep the matches that were "Completed"

rows_after_status_filter = len(matches)

print(
    "Matches removes because they were not completed:", 
    rows_before_status_filter - rows_after_status_filter
)

print("Remaining completed matches:", rows_after_status_filter)

Matches removes because they were not completed: 972
Remaining completed matches: 26602


We remove rows missing essential information. 

The required information consists of the two players, the match date, their rankings, and their ranking points.

Missing bookmaker odds are not used as a reason for removal because these variables are retained for later experiments and are not required for the current core representation.

In [10]:
# we store the number of matches before filtering for comparison reasons
rows_before_missing_filter = len(matches)

matches = matches.dropna(
    subset = essential_columns
).copy() # we remove rows with missing values inside of our essential columns

rows_after_missing_filter = len(matches)

print(
    "Matches removed because essential values were missing",
    rows_before_missing_filter - rows_after_missing_filter
)

print("Remaining matches:", rows_after_missing_filter)

Matches removed because essential values were missing 66
Remaining matches: 26536


The following check verifies that the recorded winner and loser are different players. A valid tennis match cannot have the same player in both roles, so the expected result is zero invalid rows.

In [11]:
# we check whether the values in the Winner column are equal to the values in the Loser column row by row.
same_player_rows = matches["Winner"].eq(matches["Loser"])

print(
    "Rows where winner and loser are the same player:",
    same_player_rows.sum() # returns the number of rows where winner = loser
)

Rows where winner and loser are the same player: 0


The complete rows are checked for exact duplicates. Duplicate observations could give disproportionate weight to the same match during model training

In [12]:
duplicate_count = matches.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


Two records may refer to the same match even if their remaining columns are not identical. Therefore, matches are also checked using the variables that identify a match: date, tournament, winner, and loser.

We perform a stronger check.

In [13]:
match_identifier_columns = [
    "Date", 
    "Tournament",
    "Winner",
    "Loser"
]

possible_duplicates = matches.duplicated( 
    subset = match_identifier_columns, # for each row only compares variables that identify a match
    keep = False
)

print(
    "Rows belonging to possible duplicated matches:",
    possible_duplicates.sum()
)


Rows belonging to possible duplicated matches: 0


In [14]:
## Order matches chronologically to obtain a deterministic temporal ordering.
matches = matches.sort_values (
    by = [
        "Date", 
        "ATP",
        "Tournament", 
        "Winner",
        "Loser"
    ]
).reset_index(drop=True)

matches[["Date", "Tournament", "Winner", "Loser"]].head(10)

,Date,Tournament,Winner,Loser
0,2015-01-05,Brisbane International,Chardy J.,Golubev A.
1,2015-01-05,Brisbane International,Duckworth J.,Simon G.
2,2015-01-05,Brisbane International,Kokkinakis T.,Benneteau J.
3,2015-01-05,Brisbane International,Tomic B.,Querrey S.
4,2015-01-05,Chennai Open,Coric B.,Haase R.
5,2015-01-05,Chennai Open,Muller G.,Roger-Vasselin E.
6,2015-01-05,Qatar Exxon Mobil Open,Bolelli S.,Becker B.
7,2015-01-05,Qatar Exxon Mobil Open,Brown D.,Lorenzi P.
8,2015-01-05,Qatar Exxon Mobil Open,Dodig I.,Safwat M.
9,2015-01-05,Qatar Exxon Mobil Open,Gasquet R.,Andujar P.


A reproducible neutral player ordering is created so that the identity of Player 1 is not systematically associated with the original winner.

For each match we generate a random decimal between 0 and 1, using a fixed seed (42) so that we get the same sequence of random numbers every time. 
Values below 0.5 assign the original winner to Player 1; otherwise, the original loser is assigned to Player 1. The fixed seed guarantees that the same input data and row ordering produce the same assignment when the notebook is rerun.

This transformation prevents the model from learning a trivial relationship between the dataset's player position and the target.

In [15]:
# we create a NumPy random number generator
# we use a fixed seed to make the neutral player assignment reproducible
random_generator = np.random.default_rng(seed = 42)

# for each match we generate a number from 0 to 1
player1_is_original_winner = (
    random_generator.random(len(matches)) < 0.5 
)

In [16]:
# we define context_columns with variables that describe the match itself 
# but do not depend on which player won

context_columns = [
    "ATP",
    "Location",
    "Tournament",
    "Date",
    "Series",
    "Court",
    "Surface",
    "Round",
    "Best of",
    "SourceYear"  
]

neutral_matches = matches[context_columns].copy()

In [17]:
# Map winner/loser variables to the neutral Player 1 / Player 2 representation.
neutral_matches["Player1"] = np.where(
    player1_is_original_winner, 
    matches["Winner"], # if player1_is_original_winner is True, Winner is player 1
    matches["Loser"] # otherwise, he's the Loser
)

neutral_matches["Player2"] = np.where(
    player1_is_original_winner, 
    matches["Loser"], 
    matches["Winner"]
)

In [18]:
neutral_matches["Player1Rank"] = np.where(
    player1_is_original_winner,
    matches["WRank"], 
    matches["LRank"]
)

neutral_matches["Player2Rank"] = np.where(
    player1_is_original_winner,
    matches["LRank"],
    matches["WRank"]
)

neutral_matches["Player1Points"] = np.where(
    player1_is_original_winner,
    matches["WPts"],
    matches["LPts"]
)

neutral_matches["Player2Points"] = np.where(
    player1_is_original_winner,
    matches["LPts"],
    matches["WPts"]
)

neutral_matches["Player1B365Odds"] = np.where(
    player1_is_original_winner,
    matches["B365W"],
    matches["B365L"]
)

neutral_matches["Player2B365Odds"] = np.where(
    player1_is_original_winner,
    matches["B365L"],
    matches["B365W"]
)

neutral_matches["Player1PSOdds"] = np.where(
    player1_is_original_winner,
    matches["PSW"],
    matches["PSL"]
)

neutral_matches["Player2PSOdds"] = np.where(
    player1_is_original_winner,
    matches["PSL"],
    matches["PSW"]
)

neutral_matches["Player1MaxOdds"] = np.where(
    player1_is_original_winner,
    matches["MaxW"],
    matches["MaxL"]
)

neutral_matches["Player2MaxOdds"] = np.where(
    player1_is_original_winner,
    matches["MaxL"],
    matches["MaxW"]
)

neutral_matches["Player1AvgOdds"] = np.where(
    player1_is_original_winner,
    matches["AvgW"],
    matches["AvgL"]
)

neutral_matches["Player2AvgOdds"] = np.where(
    player1_is_original_winner,
    matches["AvgL"],
    matches["AvgW"]
)

## Create target variable

The target variable Player1Won is encoded as binary value:
- 1 indicates player 1 is original winner
- 0 indicates player 2 is original winner

In [19]:
# Encode the match outcome as a binary target: 1 if Player 1 won, otherwise 0.
neutral_matches["Player1Won"] = (
    player1_is_original_winner.astype(int)
)

A unique MatchID is assigned to each observation. The columns are the reordered into a consistent structure that separates match context, player 1 variables, player 2 variables, and the target variable.

In [20]:
# Assign a unique identifier to each transformed match.
# The identifier corresponds to the row position in the transformed dataset
# it helps refer to individual matches
neutral_matches.insert(
    0, 
    "MatchID",
    np.arange(len(neutral_matches))
)

In [21]:
ordered_columns = [
    "MatchID",
    "Date",
    "SourceYear",
    "ATP",
    "Location",
    "Tournament",
    "Series",
    "Court",
    "Surface",
    "Round",
    "Best of",
    "Player1",
    "Player2",
    "Player1Rank",
    "Player2Rank",
    "Player1Points",
    "Player2Points",
    "Player1B365Odds",
    "Player2B365Odds",
    "Player1PSOdds",
    "Player2PSOdds",
    "Player1MaxOdds",
    "Player2MaxOdds",
    "Player1AvgOdds",
    "Player2AvgOdds",
    "Player1Won"   
]

neutral_matches = neutral_matches[ordered_columns]
neutral_matches.head(10)


,MatchID,Date,SourceYear,ATP,Location,Tournament,Series,Court,Surface,Round,...,Player2Points,Player1B365Odds,Player2B365Odds,Player1PSOdds,Player2PSOdds,Player1MaxOdds,Player2MaxOdds,Player1AvgOdds,Player2AvgOdds,Player1Won
0,0,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1195.0,3.50,1.28,3.50,1.34,3.50,1.36,3.30,1.32,0
1,1,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1730.0,4.50,1.18,4.67,1.23,4.73,1.23,4.31,1.20,1
2,2,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,341.0,1.44,2.62,1.53,2.67,1.53,2.80,1.47,2.62,0
3,3,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,797.0,2.25,1.57,2.37,1.65,2.37,1.67,2.25,1.61,0
4,4,2015-01-05,2015,2,Chennai,Chennai Open,ATP250,Outdoor,Hard,1st Round,...,620.0,1.72,2.00,1.75,2.18,1.80,2.25,1.72,2.07,1
5,5,2015-01-05,2015,2,Chennai,Chennai Open,ATP250,Outdoor,Hard,1st Round,...,855.0,2.10,1.66,2.10,1.81,2.15,1.81,2.05,1.73,0
6,6,2015-01-05,2015,3,Doha,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,...,810.0,1.61,2.20,1.76,2.16,1.76,2.25,1.67,2.15,0
7,7,2015-01-05,2015,3,Doha,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,...,549.0,2.37,1.53,2.55,1.57,2.62,1.60,2.40,1.54,0
8,8,2015-01-05,2015,3,Doha,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,...,168.0,1.16,5.00,1.17,5.84,1.19,5.84,1.16,5.05,1
9,9,2015-01-05,2015,3,Doha,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,...,950.0,1.20,4.33,1.24,4.44,1.26,4.84,1.21,4.15,1


**We verify that the transformation is correct**

The code reconstructs the winner from the transformed table:
- when Player1Won = 1, the winner shouldd be Player 1;
- when Player1Won = 0, the winner shouldd be Player 2.

It then compares the reconstructed winner with the original Winner column.

If the result is True, every row was mapped consistently.

In [22]:
# we reconstruct the original winner from the new neutral representation
# if Player1Won = 1 then reconstructed_winner = Player1
reconstructed_winner = np.where(
    neutral_matches["Player1Won"].eq(1),
    neutral_matches["Player1"],
    neutral_matches["Player2"]
)

# we compare the reconstructed winner with the original Winner column.
transformation_is_correct = (
    reconstructed_winner == matches["Winner"].to_numpy()
).all() # checks whether all values are true

print(
    "Every winner was reconstructed correctly.",
    transformation_is_correct
)

Every winner was reconstructed correctly. True


In [23]:
assert neutral_matches["Player1"].ne(
    neutral_matches["Player2"]
).all()

assert set(
    neutral_matches["Player1Won"].unique()
) == {0, 1}

print("Basic validation tests passed.")

Basic validation tests passed.


We check the class distribution of Player1Won to ensure there's no class imbalance. 
The two classes should occur in aproximately equal proportions.

In [24]:
target_counts = neutral_matches["Player1Won"].value_counts()
target_percentages = (
    neutral_matches["Player1Won"]
    .value_counts(normalize = True) # returns proportion of matches
    .sort_index() # sorts by class 0 or 1 for consistency
    .mul(100) # converts proportions to percentages
)

print("Target counts:")
print(target_counts)

print("\nTarget percentages:")
print(target_percentages)

Target counts:
Player1Won
0    13334
1    13202
Name: count, dtype: int64

Target percentages:
Player1Won
0    50.248719
1    49.751281
Name: proportion, dtype: float64


In [25]:
# we construct the path and filename where the processed dataset will be saved.
output_file = (
    PROCESSED_DATA_DIR
    / "matches_neutral_2015_2025.csv"
)

neutral_matches.to_csv(
    output_file, # writes dataframe to a CSV file at the path stored in output_file 
    index=False # does not write the DataFrame index as an additional CSV column
)

print("Dataset saved to:")
print(output_file.resolve())

Dataset saved to:
C:\Users\SegreteriaCampiello\Documents\GitHub\artificial_intelligence\data\processed\matches_neutral_2015_2025.csv


In [26]:
# verify that the saved file can be loaded for subsequent notebooks 
saved_matches = pd.read_csv(
    output_file, 
    parse_dates=["Date"] # restore date as a datetime rather than plain text
)

print("Saved dataset shape:", saved_matches.shape)
print("First date: ", saved_matches["Date"].min())
print("Last date", saved_matches["Date"].max())

saved_matches.head()

Saved dataset shape: (26536, 26)
First date:  2015-01-05 00:00:00
Last date 2025-11-16 00:00:00


,MatchID,Date,SourceYear,ATP,Location,Tournament,Series,Court,Surface,Round,...,Player2Points,Player1B365Odds,Player2B365Odds,Player1PSOdds,Player2PSOdds,Player1MaxOdds,Player2MaxOdds,Player1AvgOdds,Player2AvgOdds,Player1Won
0,0,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1195.0,3.50,1.28,3.50,1.34,3.50,1.36,3.30,1.32,0
1,1,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1730.0,4.50,1.18,4.67,1.23,4.73,1.23,4.31,1.20,1
2,2,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,341.0,1.44,2.62,1.53,2.67,1.53,2.80,1.47,2.62,0
3,3,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,797.0,2.25,1.57,2.37,1.65,2.37,1.67,2.25,1.61,0
4,4,2015-01-05,2015,2,Chennai,Chennai Open,ATP250,Outdoor,Hard,1st Round,...,620.0,1.72,2.00,1.75,2.18,1.80,2.25,1.72,2.07,1


In [27]:
print("Final shape:", neutral_matches.shape)
print(neutral_matches["Player1Won"].value_counts(normalize=True))
print("First date:", neutral_matches["Date"].min())
print("Last date", neutral_matches["Date"].max())


Final shape: (26536, 26)
Player1Won
0    0.502487
1    0.497513
Name: proportion, dtype: float64
First date: 2015-01-05 00:00:00
Last date 2025-11-16 00:00:00
